# SU(3) Wilson rare hard-defect Peierls probe v2

Run the single code cell below. This version sweeps the defect threshold `delta`, avoids huge tail printouts, and looks for the actual rare-defect Peierls regime.

In [ ]:

# SU(3) Wilson RARE hard-defect Peierls probe v2
#
# Why this exists:
#   The first rooted-defect run used delta=0.35. At beta=4.8--6.0 this marked
#   55%--80% of plaquettes as "defects", so the defect set percolated. That is
#   not the Peierls regime. This v2 scan sweeps delta and reports only compact
#   threshold diagnostics.
#
# What this tests:
#   P_beta(Gamma subset D_delta) <= K^|Gamma| exp(-alpha beta delta |Gamma|)
#
# It prints:
#   - defect density rho_delta
#   - P(root bad)
#   - max rooted cluster size
#   - fixed-animal inclusion exponents alpha_eff_n
#   - alpha beta delta margins against a crude animal-counting threshold
#
# No files are written.

import math, time
from collections import defaultdict, deque
import numpy as np
import matplotlib.pyplot as plt

# =============================================================================
# PARAMETERS
# =============================================================================
L = 4
BETAS = [5.6, 6.0, 6.4, 6.8]        # push slightly upward from the failed run
DELTAS = [0.35, 0.50, 0.70, 0.90, 1.10, 1.30]
THERM_SWEEPS = 120
MEAS = 100
SWEEPS_BETWEEN = 4
EPS0 = 0.22
SEED = 20260611

MAX_ANIMAL_N = 10

ACC_LOW, ACC_HIGH = 0.35, 0.65
EPS_UP, EPS_DOWN = 1.06, 0.94

rng = np.random.default_rng(SEED)
D = 4
NC = 3
I3 = np.eye(3, dtype=np.complex128)
ORIS = tuple((i, j) for i in range(D) for j in range(i + 1, D))
NPLAQS = (L**D) * len(ORIS)

print("=" * 104)
print("SU(3) WILSON RARE HARD-DEFECT PEIERLS PROBE v2")
print("=" * 104)
print(f"L={L}, volume={L**D}, plaquettes={NPLAQS}")
print(f"betas={BETAS}")
print(f"deltas={DELTAS}")
print(f"therm={THERM_SWEEPS}, measurements={MEAS}, sweeps_between={SWEEPS_BETWEEN}, seed={SEED}")

# =============================================================================
# LATTICE / GROUP UTILITIES
# =============================================================================
def shift(x, mu, s=1):
    y = list(x)
    y[mu] = (y[mu] + s) % L
    return tuple(y)

def matdag(A):
    return A.conj().T

def project_su3(M):
    u, _, vh = np.linalg.svd(M)
    P = u @ vh
    det = np.linalg.det(P)
    P = P / det**(1.0 / 3.0)
    return P.astype(np.complex128)

def random_su3_near_identity(eps):
    A = rng.normal(size=(3, 3)) + 1j * rng.normal(size=(3, 3))
    H = (A + matdag(A)) / 2.0
    H = H - np.trace(H) * I3 / 3.0
    nrm = math.sqrt(float(np.real(np.trace(H @ H))))
    if nrm < 1e-14:
        return I3.copy()
    H = H / nrm
    vals, vecs = np.linalg.eigh(H)
    R = vecs @ np.diag(np.exp(1j * eps * vals)) @ matdag(vecs)
    return project_su3(R)

def cold_start():
    U = np.empty((L, L, L, L, D, 3, 3), dtype=np.complex128)
    U[...] = I3
    return U

def get_link(U, x, mu):
    return U[x + (mu,)]

def plaquette_matrix(U, x, ori):
    mu, nu = ori
    x_mu = shift(x, mu, +1)
    x_nu = shift(x, nu, +1)
    return (
        get_link(U, x, mu)
        @ get_link(U, x_mu, nu)
        @ matdag(get_link(U, x_nu, mu))
        @ matdag(get_link(U, x, nu))
    )

def plaquette_energy(U, x, ori):
    Up = plaquette_matrix(U, x, ori)
    return 1.0 - float(np.real(np.trace(Up))) / 3.0

def plaquette_links(plaq):
    x, oi = plaq
    mu, nu = ORIS[oi]
    return {
        (x, mu),
        (shift(x, mu, +1), nu),
        (shift(x, nu, +1), mu),
        (x, nu),
    }

PLAQS = []
for x in np.ndindex(L, L, L, L):
    for oi in range(len(ORIS)):
        PLAQS.append((tuple(x), oi))

LINK_TO_PLAQS = defaultdict(set)
for p in PLAQS:
    for ell in plaquette_links(p):
        LINK_TO_PLAQS[ell].add(p)

NEIGH = {}
for p in PLAQS:
    ns = set()
    for ell in plaquette_links(p):
        ns |= LINK_TO_PLAQS[ell]
    ns.discard(p)
    NEIGH[p] = frozenset(ns)

ROOT = ((0, 0, 0, 0), 0)
ROOT_DEG = len(NEIGH[ROOT])
MU_CRUDE = math.e * ROOT_DEG

print(f"Root plaquette={ROOT}, orientation={ORIS[ROOT[1]]}, adjacency degree={ROOT_DEG}")
print(f"Crude rooted-animal factor mu <= e*Delta = {MU_CRUDE:.6f}, log(mu)={math.log(MU_CRUDE):.6f}")

def containing_plaquettes_for_link(x, mu):
    target = (x, mu)
    out = set()
    for nu in range(D):
        if nu == mu:
            continue
        oi = ORIS.index(tuple(sorted((mu, nu))))
        for base in (x, shift(x, nu, -1)):
            p = (base, oi)
            if target in plaquette_links(p):
                out.add(p)
    return tuple(out)

CONTAINING = {}
for x in np.ndindex(L, L, L, L):
    x = tuple(x)
    for mu in range(D):
        ps = containing_plaquettes_for_link(x, mu)
        assert len(ps) == 2 * (D - 1), (x, mu, len(ps))
        CONTAINING[(x, mu)] = ps

def local_energy_for_link(U, x, mu):
    return sum(plaquette_energy(U, base, ORIS[oi]) for base, oi in CONTAINING[(x, mu)])

def sweep(U, beta, eps):
    accepted = 0
    total = 0
    for x in np.ndindex(L, L, L, L):
        x = tuple(x)
        for mu in range(D):
            total += 1
            old = U[x + (mu,)].copy()
            e_old = local_energy_for_link(U, x, mu)

            R = random_su3_near_identity(eps)
            U[x + (mu,)] = project_su3(R @ old)
            e_new = local_energy_for_link(U, x, mu)

            dS = beta * (e_new - e_old)
            if dS <= 0.0 or rng.random() < math.exp(-dS):
                accepted += 1
            else:
                U[x + (mu,)] = old
    return accepted / total

def all_plaquette_energies(U):
    return np.array([plaquette_energy(U, base, ORIS[oi]) for base, oi in PLAQS], dtype=float)

ROOT_INDEX = PLAQS.index(ROOT)

def rooted_cluster_size(defect_bool):
    if not defect_bool[ROOT_INDEX]:
        return 0
    seen = {ROOT}
    q = deque([ROOT])
    while q:
        p = q.popleft()
        for nb in NEIGH[p]:
            j = PLAQS.index(nb)
            if defect_bool[j] and nb not in seen:
                seen.add(nb)
                q.append(nb)
    return len(seen)

# Precompute neighbor indices to avoid repeated PLAQS.index in cluster search.
PLAQ_INDEX = {p: i for i, p in enumerate(PLAQS)}
NEIGH_IDX = {PLAQ_INDEX[p]: [PLAQ_INDEX[q] for q in NEIGH[p]] for p in PLAQS}

def rooted_cluster_size_fast(defect_bool):
    if not defect_bool[ROOT_INDEX]:
        return 0
    seen = {ROOT_INDEX}
    q = deque([ROOT_INDEX])
    while q:
        p = q.popleft()
        for nb in NEIGH_IDX[p]:
            if defect_bool[nb] and nb not in seen:
                seen.add(nb)
                q.append(nb)
    return len(seen)

def grow_fixed_root_animals(max_n):
    animals = {1: frozenset([ROOT_INDEX])}
    current = {ROOT}
    for n in range(2, max_n + 1):
        boundary = set()
        for p in current:
            boundary |= set(NEIGH[p])
        boundary -= current
        if not boundary:
            break
        q = sorted(boundary, key=lambda p: (sum(min(t, L-t) for t in p[0]), p[1], p[0]))[0]
        current = set(current)
        current.add(q)
        animals[n] = frozenset(PLAQ_INDEX[p] for p in current)
    return animals

FIXED_ANIMALS = grow_fixed_root_animals(MAX_ANIMAL_N)
print("Fixed rooted animals:", sorted(FIXED_ANIMALS.keys()))

# =============================================================================
# ANALYSIS OF A MEASUREMENT SET
# =============================================================================
def analyze_samples(beta, energy_samples):
    E = np.asarray(energy_samples, dtype=float)  # shape MEAS x NPLAQS
    out = {}

    print("\n" + "=" * 104)
    print(f"BETA={beta:.4f} ANALYSIS")
    print("=" * 104)

    meanV = float(np.mean(E))
    print(f"mean V = {meanV:.8f}")
    print("plaquette-energy quantiles:")
    for q in [0.50, 0.70, 0.80, 0.90, 0.95, 0.97, 0.99, 0.995]:
        print(f"  q={q:5.3f}: {np.quantile(E, q):.6f}")

    print("\nAbsolute-threshold defect results:")
    print("delta   rho_delta       P(root)      max|Croot|  alpha_root  alpha*beta*delta   margin_vs_logmu")
    for delta in DELTAS:
        defect = E >= delta
        rho = float(np.mean(defect))
        root_series = defect[:, ROOT_INDEX]
        p_root = float(np.mean(root_series))
        p_root_lap = (float(np.sum(root_series)) + 0.5) / (len(root_series) + 1.0)

        sizes = np.array([rooted_cluster_size_fast(row) for row in defect], dtype=int)
        maxC = int(np.max(sizes))
        alpha_root = -math.log(max(p_root_lap, 1e-300)) / (beta * delta)
        lhs = alpha_root * beta * delta
        margin = lhs - math.log(MU_CRUDE)

        print(f"{delta:5.2f}   {rho:12.8f}  {p_root:12.8f}  {maxC:10d}  "
              f"{alpha_root:10.6f}  {lhs:16.6f}  {margin:+16.6f}")

        fixed_alpha = {}
        fixed_prob = {}
        for n, A in FIXED_ANIMALS.items():
            hits = sum(1 for row in defect if all(row[j] for j in A))
            p_lap = (hits + 0.5) / (len(defect) + 1.0)
            fixed_prob[n] = hits / len(defect)
            fixed_alpha[n] = -math.log(max(p_lap, 1e-300)) / (beta * delta * n)

        out[delta] = {
            "rho": rho,
            "p_root": p_root,
            "p_root_lap": p_root_lap,
            "maxC": maxC,
            "alpha_root": alpha_root,
            "lhs": lhs,
            "margin": margin,
            "fixed_alpha": fixed_alpha,
            "fixed_prob": fixed_prob,
            "cluster_sizes": sizes,
        }

    # Compact fixed-animal table for the first rare-ish delta values.
    print("\nFixed-animal exponent table for rare regimes only:")
    for delta in DELTAS:
        if out[delta]["rho"] <= 0.20 or delta == DELTAS[-1]:
            print(f"\ndelta={delta:.2f}, rho={out[delta]['rho']:.6f}, maxC={out[delta]['maxC']}")
            print("n     P_hat       alpha_eff_n")
            for n in sorted(FIXED_ANIMALS):
                print(f"{n:2d}  {out[delta]['fixed_prob'][n]:10.6f}  {out[delta]['fixed_alpha'][n]:12.6f}")

    return out

# =============================================================================
# MONTE CARLO RUN
# =============================================================================
def run_beta(beta):
    U = cold_start()
    eps = EPS0

    print("\n" + "-" * 104)
    print(f"beta={beta:.4f}")
    print("-" * 104)

    # thermalize with adaptation
    recent_mean = []
    for t in range(1, THERM_SWEEPS + 1):
        acc = sweep(U, beta, eps)
        if acc < ACC_LOW:
            eps *= EPS_DOWN
        elif acc > ACC_HIGH:
            eps *= EPS_UP

        if t in {1, 5, 10, 20, 40, 80, THERM_SWEEPS}:
            E = all_plaquette_energies(U)
            recent_mean.append(float(np.mean(E)))
            print(f"therm {t:4d}: acc={acc:.3f}, eps={eps:.4f}, meanV={np.mean(E):.6f}, "
                  f"q95={np.quantile(E,0.95):.6f}, q99={np.quantile(E,0.99):.6f}")

    samples = []
    for m in range(1, MEAS + 1):
        for _ in range(SWEEPS_BETWEEN):
            sweep(U, beta, eps)
        E = all_plaquette_energies(U)
        samples.append(E)
        if m in {1, 10, 25, 50, MEAS}:
            print(f"meas {m:4d}: meanV={np.mean(E):.6f}, q95={np.quantile(E,0.95):.6f}, q99={np.quantile(E,0.99):.6f}")

    return analyze_samples(beta, np.array(samples))

all_results = {}
for beta in BETAS:
    all_results[beta] = run_beta(beta)

# =============================================================================
# CROSS-BETA PLOTS
# =============================================================================
plt.figure(figsize=(8.5, 5.2))
for delta in DELTAS:
    xs, ys = [], []
    for beta in BETAS:
        xs.append(beta)
        ys.append(all_results[beta][delta]["rho"])
    plt.plot(xs, ys, "o-", label=f"delta={delta:.2f}")
plt.yscale("log")
plt.xlabel("Wilson beta")
plt.ylabel("defect density rho_delta")
plt.title("Hard-defect density vs beta and delta")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

plt.figure(figsize=(8.5, 5.2))
for delta in DELTAS:
    xs, ys = [], []
    for beta in BETAS:
        xs.append(beta)
        ys.append(all_results[beta][delta]["lhs"])
    plt.plot(xs, ys, "o-", label=f"delta={delta:.2f}")
plt.axhline(math.log(MU_CRUDE), linestyle="--", label="log(mu_crude)")
plt.xlabel("Wilson beta")
plt.ylabel("alpha_root * beta * delta = -log P(root)")
plt.title("Root-event Peierls threshold diagnostic")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

plt.figure(figsize=(8.5, 5.2))
for delta in DELTAS:
    xs, ys = [], []
    for beta in BETAS:
        xs.append(beta)
        ys.append(all_results[beta][delta]["maxC"])
    plt.plot(xs, ys, "o-", label=f"delta={delta:.2f}")
plt.xlabel("Wilson beta")
plt.ylabel("max rooted cluster size observed")
plt.title("Rooted cluster percolation check")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

print("\n" + "=" * 104)
print("VERDICT RULE")
print("=" * 104)
print("""
The previous delta=0.35 run was outside the Peierls regime because rho_delta was
O(1) and rooted clusters percolated.

A useful regime in this v2 scan is one where:
  1. rho_delta is small, preferably < 0.05 or at least < 0.10;
  2. max rooted cluster sizes are not O(volume);
  3. fixed-animal probabilities decay clearly with n;
  4. alpha_root*beta*delta moves toward or above log(mu_crude).

The last condition is deliberately severe because mu_crude=e*Delta is a loose
proof-friendly animal bound. If it fails only by a modest constant, the next move
is to improve the animal-counting/capacity bound. If it fails by orders of
magnitude, the hard-defect Peierls route at that delta is not viable.
""")
